# 🎭 Step 2.5: ACTスタイル模倣学習 Fine-tune

PPO学習済みポリシーに **ブラウザのキーボード操作デモ** を使って
ACTスタイル（アクションチャンキング）で追加学習します。

## BCとACTの違い

```
BC (Behavior Cloning):
  観測(t) → 行動(t)  ← 1ステップずつ独立予測
  問題: 少しずれると「見たことない状態」に陥る (分布シフト)

ACT (Action Chunking with Transformers):
  観測(t) → [行動(t), 行動(t+1), ..., 行動(t+K-1)]
              ↑ K=10 個の行動をまとめて予測
  → 滑らかで一貫性のある動き
  → 分布シフトの影響を受けにくい
  → LeRobotで使っているのと同じ方式
```

## デモデータの準備 (step3_persona_city_sim.htmlで収集)

```
1. python3 -m http.server 8000
2. http://localhost:8000/step3_persona_city_sim.html を開く
3. 記録したいペルソナのカードをクリックして選択
4. [⏺ Record] ボタンをクリック
   → キーボード操作モードが有効になる
      W / ↑ : 前進
      A / ← : 左回転
      D / → : 右回転
5. 「こういう動きがこのペルソナらしい」という経路を操作
6. [⏹ Stop] で停止 → エピソードとして保存
7. 4〜6を繰り返す (10〜30エピソード推奨)
8. [⬇ Export Demo] でJSONをダウンロード
9. Google Driveにアップロード
```

## アーキテクチャ

```
観測: FPV画像 → DINOv2 → CLS(384) + 建物分類(8) = 392次元
        ↓
   ACTPolicy (Transformer)
        ↓
   [行動(t), ..., 行動(t+K-1)]  K=10個

実行: K個の行動を順番に実行 → K ステップ後に再推論
```

In [ ]:
# ============================================================
# セル 1: インストール
# ============================================================
!pip install torch torchvision onnx onnxscript onnxruntime numpy -q
print('✓ Install complete')

In [ ]:
# ============================================================
# セル 2: インポート & 設定
# ============================================================
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, os, json, math, time, random
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams.update({
    'figure.facecolor':'#0a0c10','axes.facecolor':'#12161e',
    'text.color':'#c8d8e8',
})
from IPython.display import clear_output
import torchvision.transforms as T

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/mesa_persona'
ONNX_DIR = '/content/drive/MyDrive/mesa_persona_onnx'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(ONNX_DIR, exist_ok=True)

# ── ACT設定 ──
TARGET_PERSONA_ID = 'A'   # fine-tune するペルソナ ('A'〜'E')
CHUNK_SIZE        = 10    # K: 何ステップ先まで予測するか
D_MODEL           = 256   # Transformerの隠れ次元
N_HEADS           = 4     # Attention ヘッド数
N_LAYERS          = 2     # Transformer層数
ACT_LR            = 1e-4
ACT_EPOCHS        = 100
BATCH_SIZE        = 32

# step2 と一致させる定数
IMG_W=224; IMG_H=224; IMG_CH=3
DINO_MODEL='dinov2_vits14'; DINO_DIM=384
N_BLDG_CLASSES = 8
OBS_DIM = DINO_DIM + N_BLDG_CLASSES  # 392
ACT_DIM = 3   # 前進/左回転/右回転

print(f'CHUNK_SIZE: {CHUNK_SIZE}  OBS_DIM: {OBS_DIM}')
print()
print('⚠️  実行前に step2_persona_train.ipynb のセル1〜8を実行してください')
print('   (dino_backbone, clf_sess 等が定義済みである必要があります)')

In [ ]:
# ============================================================
# セル 3: デモJSONの読み込み
# ============================================================

# ブラウザからエクスポートしたJSONのパス
# ファイル名は mesa_demo_XXXXXXXXXX.json 形式
# Google Drive の mesa_persona/ フォルダに置いてください
DEMO_JSON_PATH = f'{SAVE_DIR}/mesa_demo_latest.json'

# 複数ファイルを使う場合はここに追加
DEMO_JSON_PATHS = [
    DEMO_JSON_PATH,
    # f'{SAVE_DIR}/mesa_demo_session2.json',
]

if not os.path.exists(DEMO_JSON_PATH):
    print(f'❌ {DEMO_JSON_PATH} が見つかりません')
    print()
    print('準備手順:')
    print('  1. step3_persona_city_sim.html でキーボード操作')
    print('  2. [⬇ Export Demo] でJSONをダウンロード')
    print(f'  3. {SAVE_DIR}/ に mesa_demo_latest.json として配置')
else:
    raw_episodes = []
    for path in DEMO_JSON_PATHS:
        if not os.path.exists(path): continue
        with open(path) as f:
            data = json.load(f)
        eps = [e for e in data if e.get('persona_id') == TARGET_PERSONA_ID]
        raw_episodes.extend(eps)
        print(f'✓ {os.path.basename(path)}: {len(eps)}エピソード (persona={TARGET_PERSONA_ID})')

    if not raw_episodes:
        print(f'❌ persona_id={TARGET_PERSONA_ID} のデモがありません')
    else:
        lengths = [e['total_steps'] for e in raw_episodes]
        print(f'\n合計: {len(raw_episodes)}エピソード')
        print(f'ステップ数: 最小={min(lengths)}  最大={max(lengths)}  '
              f'平均={sum(lengths)/len(lengths):.0f}')

        from collections import Counter
        all_acts = [s['action'] for e in raw_episodes for s in e['steps']]
        ac = Counter(all_acts); total_ac = len(all_acts)
        print(f'行動分布: 前進={ac[0]/total_ac:.1%}  '
              f'左={ac[1]/total_ac:.1%}  右={ac[2]/total_ac:.1%}')

In [ ]:
# ============================================================
# セル 4: DINOv2特徴抽出 & ACTデータセット作成
# ============================================================

class ACTDataset(Dataset):
    """
    デモエピソードを (obs_feat, action_chunk) ペアに変換。

    obs_feat:     (OBS_DIM,)  DINOv2 CLS + 建物分類スコア
    action_chunk: (K,)        次のK個の行動インデックス
    """
    def __init__(self, episodes, chunk_size):
        self.samples = []
        dino_mean = torch.tensor([0.485,0.456,0.406], device=DEVICE).view(1,3,1,1)
        dino_std  = torch.tensor([0.229,0.224,0.225], device=DEVICE).view(1,3,1,1)

        print(f'特徴抽出中 ({len(episodes)}エピソード)...')
        for ep_idx, ep in enumerate(episodes):
            steps   = ep['steps']
            T_steps = len(steps)
            if T_steps < chunk_size + 1: continue

            # FPV画像 → DINOv2特徴 を一括抽出
            ep_feats = []
            for step in steps:
                fp_arr  = np.array(step['obs_fp'], dtype=np.float32)
                fp_t    = torch.tensor(fp_arr).view(IMG_CH,IMG_H,IMG_W).to(DEVICE)
                fp_norm = (fp_t.unsqueeze(0) - dino_mean) / dino_std

                with torch.no_grad():
                    out  = dino_backbone.forward_features(fp_norm)
                    feat = out['x_norm_clstoken'].squeeze(0)  # (384,)

                # 建物分類スコアを追加
                feat_np = feat.cpu().numpy().reshape(1,-1).astype(np.float32)
                if clf_sess is not None:
                    lg  = clf_sess.run(None,{'dino_feat':feat_np})[0]
                    exp = np.exp(lg - lg.max())
                    probs = (exp/exp.sum()).flatten()
                else:
                    probs = np.zeros(N_BLDG_CLASSES, dtype=np.float32)

                combined = np.concatenate([feat.cpu().numpy(), probs])  # (392,)
                ep_feats.append(combined)

            ep_feats_np = np.stack(ep_feats)   # (T, 392)
            actions_np  = np.array([s['action'] for s in steps], dtype=np.int64)

            # チャンク化
            for t in range(T_steps - chunk_size):
                obs_feat     = torch.tensor(ep_feats_np[t],           dtype=torch.float32)
                action_chunk = torch.tensor(actions_np[t:t+chunk_size], dtype=torch.long)
                self.samples.append((obs_feat, action_chunk))

            if (ep_idx+1) % 5 == 0:
                print(f'  {ep_idx+1}/{len(episodes)}  サンプル数: {len(self.samples)}')

        print(f'✓ {len(self.samples)}サンプル生成')

    def __len__(self): return len(self.samples)
    def __getitem__(self, i): return self.samples[i]


dataset = ACTDataset(raw_episodes, CHUNK_SIZE)

n_train = int(len(dataset)*0.85)
n_val   = len(dataset) - n_train
train_ds, val_ds = torch.utils.data.random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
print(f'Train:{len(train_ds)}  Val:{len(val_ds)}  Batch:{BATCH_SIZE}')

In [ ]:
# ============================================================
# セル 5: ACTPolicy モデル定義
# ============================================================

class ACTPolicy(nn.Module):
    """
    ACT (Action Chunking with Transformers) ポリシー。

    入力:  obs_feat  (B, OBS_DIM=392)
    出力:  logits    (B, K, ACT_DIM=3)

    推論時: K個の行動を argmax で決定し、順番に実行する。
    """
    def __init__(self, obs_dim=OBS_DIM, act_dim=ACT_DIM,
                 chunk_size=CHUNK_SIZE, d_model=D_MODEL,
                 n_heads=N_HEADS, n_layers=N_LAYERS):
        super().__init__()
        self.chunk_size = chunk_size

        # 観測 → d_model 次元への射影
        self.obs_embed = nn.Sequential(
            nn.Linear(obs_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
        )
        # チャンク内の位置埋め込み (K個のステップを区別する)
        self.pos_embed = nn.Embedding(chunk_size, d_model)

        # Transformer Encoder
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model*4,
            dropout=0.1, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_layers)

        # 各タイムステップの行動を予測
        self.action_head = nn.Linear(d_model, act_dim)

    def forward(self, obs_feat):
        """
        obs_feat: (B, OBS_DIM)
        returns:  (B, K, ACT_DIM)
        """
        B = obs_feat.shape[0]
        K = self.chunk_size

        obs_emb = self.obs_embed(obs_feat)             # (B, d_model)
        pos_idx = torch.arange(K, device=obs_feat.device)
        pos_emb = self.pos_embed(pos_idx).unsqueeze(0).expand(B,-1,-1)  # (B,K,d)

        # 観測 + 位置埋め込みを K 個のクエリとして使用
        queries = pos_emb + obs_emb.unsqueeze(1)       # (B, K, d_model)
        hidden  = self.transformer(queries)             # (B, K, d_model)
        return self.action_head(hidden)                 # (B, K, ACT_DIM)

    @torch.no_grad()
    def predict_chunk(self, obs_feat):
        """推論: K個の行動インデックスを返す (B, K)"""
        return self.forward(obs_feat).argmax(dim=-1)


act_policy = ACTPolicy().to(DEVICE)
n_params   = sum(p.numel() for p in act_policy.parameters())
print(f'ACTPolicy: {n_params:,} params')

dummy = torch.randn(4, OBS_DIM, device=DEVICE)
out   = act_policy(dummy)
print(f'forward OK: {dummy.shape} → {out.shape}')
print(f'  期待値: (4, {CHUNK_SIZE}, {ACT_DIM})')
del dummy, out

In [ ]:
# ============================================================
# セル 6: ACT 学習
# ============================================================
optimizer = torch.optim.AdamW(act_policy.parameters(), lr=ACT_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=ACT_EPOCHS)

act_ckpt_path = f'{SAVE_DIR}/act_{TARGET_PERSONA_ID}.pt'
best_val_loss = float('inf')
best_state    = None
train_losses, val_losses, val_accs = [], [], []

print(f'ACT学習開始  Epochs:{ACT_EPOCHS}  K={CHUNK_SIZE}')

for epoch in range(ACT_EPOCHS):
    # Train
    act_policy.train()
    ep_loss = 0.0
    for obs_feat, act_chunk in train_dl:
        obs_feat  = obs_feat.to(DEVICE)    # (B, 392)
        act_chunk = act_chunk.to(DEVICE)   # (B, K)
        logits    = act_policy(obs_feat)   # (B, K, 3)
        B,K,C     = logits.shape
        loss      = F.cross_entropy(logits.reshape(B*K,C), act_chunk.reshape(B*K))
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(act_policy.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()
    scheduler.step()
    ep_loss /= len(train_dl)

    # Validation
    act_policy.eval()
    v_loss=0.0; correct=0; total_v=0
    with torch.no_grad():
        for obs_feat, act_chunk in val_dl:
            obs_feat = obs_feat.to(DEVICE); act_chunk = act_chunk.to(DEVICE)
            logits   = act_policy(obs_feat)
            B,K,C    = logits.shape
            v_loss  += F.cross_entropy(logits.reshape(B*K,C),act_chunk.reshape(B*K)).item()
            correct += (logits.argmax(-1)==act_chunk).sum().item()
            total_v += B*K
    v_loss /= len(val_dl); v_acc = correct/total_v

    train_losses.append(ep_loss); val_losses.append(v_loss); val_accs.append(v_acc)
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_state    = {k:v.clone() for k,v in act_policy.state_dict().items()}

    if (epoch+1)%20==0 or epoch==0:
        clear_output(wait=True)
        fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(12,3))
        fig.patch.set_facecolor('#0a0c10')
        for ax,(y,c,t) in zip([ax1,ax2,ax3],[
            (train_losses,'#ff5050','Train Loss'),
            (val_losses,  '#4488ff','Val Loss'),
            (val_accs,    '#00ddb4','Val Accuracy'),
        ]):
            ax.set_facecolor('#12161e')
            if len(y)>=2: ax.plot(y,color=c,lw=1.5)
            ax.set_title(f'{t}\n{y[-1]:.4f}' if y else t,color='#c8d8e8',fontsize=9)
            ax.grid(color='#1e2530',lw=0.5); ax.spines[:].set_color('#1e2530')
        if val_accs: ax3.set_ylim(0,1)
        fig.suptitle(
            f'ACT [{TARGET_PERSONA_ID}]  Epoch:{epoch+1}/{ACT_EPOCHS}  '
            f'Best:{best_val_loss:.4f}  ValAcc:{v_acc:.1%}',
            color='#00ddb4',fontsize=11)
        plt.tight_layout(); plt.show()
        print(f'Epoch{epoch+1:4d} | Train:{ep_loss:.4f} | '
              f'Val:{v_loss:.4f} | Acc:{v_acc:.1%}')

act_policy.load_state_dict(best_state)
torch.save({'model':best_state,'chunk_size':CHUNK_SIZE}, act_ckpt_path)
print(f'\n✓ 学習完了  Best ValLoss:{best_val_loss:.4f}')

In [ ]:
# ============================================================
# セル 7: ACT ONNX エクスポート
# ============================================================
import onnx, onnxruntime as ort

act_policy.eval().cpu()
onnx_path = f'{ONNX_DIR}/persona_{TARGET_PERSONA_ID}_act.onnx'
dummy     = torch.zeros(1, OBS_DIM)

with torch.no_grad():
    torch.onnx.export(
        act_policy, dummy, onnx_path,
        input_names=['obs_feat'],
        output_names=['action_logits'],
        dynamic_axes={'obs_feat':{0:'batch'},'action_logits':{0:'batch'}},
        opset_version=17,
    )
onnx_model = onnx.load(onnx_path)
onnx.save(onnx_model, onnx_path, save_as_external_data=False)
print(f'✓ {os.path.basename(onnx_path)} ({os.path.getsize(onnx_path)/1e6:.2f}MB)')

# 推論テスト
sess   = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
dummy_n= np.zeros((1,OBS_DIM),dtype=np.float32)
out    = sess.run(None,{'obs_feat':dummy_n})[0]
print(f'✓ 推論テスト OK  出力shape: {out.shape}')
print(f'  = (1, {CHUNK_SIZE}チャンク, {ACT_DIM}行動)')

# メタデータ
with open(f'{SAVE_DIR}/persona_rewards.json') as f:
    rp = json.load(f)[TARGET_PERSONA_ID]

meta = {
    'persona_id':      TARGET_PERSONA_ID,
    'persona_name':    rp.get('persona_name',''),
    'model_type':      'act',
    'chunk_size':      CHUNK_SIZE,
    'obs_dim':         OBS_DIM,
    'act_dim':         ACT_DIM,
    'input_name':      'obs_feat',
    'output_name':     'action_logits',
    'img_w':IMG_W, 'img_h':IMG_H, 'img_ch':IMG_CH,
    'dino_dim':        DINO_DIM,
    'n_bldg_classes':  N_BLDG_CLASSES,
    'n_demo_episodes': len(raw_episodes),
    'best_val_loss':   float(best_val_loss),
}
meta_path = onnx_path.replace('.onnx','_meta.json')
with open(meta_path,'w') as f:
    json.dump(meta,f,indent=2,ensure_ascii=False)
print(f'✓ メタデータ: {os.path.basename(meta_path)}')

print()
print('='*50)
print(f'✅ Step 2.5 完了!')
print(f'   persona_{TARGET_PERSONA_ID}_act.onnx')
print(f'   → data/ に配置してHTMLで使用')
act_policy.to(DEVICE)

In [ ]:
# ============================================================
# セル 8: デモ vs ACT予測 の行動分布比較
# ============================================================
from collections import Counter

demo_acts  = [s['action'] for e in raw_episodes for s in e['steps']]
demo_dist  = Counter(demo_acts); total_d = len(demo_acts)

act_policy.eval()
pred_acts = []
with torch.no_grad():
    for obs_feat,_ in val_dl:
        preds = act_policy(obs_feat.to(DEVICE)).argmax(-1).flatten().cpu().tolist()
        pred_acts.extend(preds)
pred_dist  = Counter(pred_acts); total_p = len(pred_acts)

labels = ['前進','左回転','右回転']
print(f'[{TARGET_PERSONA_ID}] 行動分布比較:')
print(f'  {'':10} {'デモ':>10} {'ACT予測':>10}')
for i,n in enumerate(labels):
    d = demo_dist[i]/total_d; p = pred_dist[i]/total_p
    print(f'  {n:10} {d:>9.1%} {p:>9.1%}')

x=np.arange(3); w=0.35
fig,ax=plt.subplots(figsize=(7,4))
fig.patch.set_facecolor('#0a0c10'); ax.set_facecolor('#12161e')
ax.bar(x-w/2,[demo_dist[i]/total_d for i in range(3)],w,label='デモ',   color='#ffc840')
ax.bar(x+w/2,[pred_dist[i]/total_p for i in range(3)],w,label='ACT予測',color='#00ddb4')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0,1); ax.legend()
ax.set_title(f'[{TARGET_PERSONA_ID}] デモ vs ACT予測\nVal Acc={val_accs[-1]:.1%}',
             color='#c8d8e8')
ax.spines[:].set_color('#1e2530'); ax.tick_params(colors='#c8d8e8')
plt.tight_layout(); plt.show()